# Stream a Response via our Google Platform

This example shows how to stream a response from the Google platform using our platform client.

## Setup


In [ ]:
import os

from utils import setup_notebook

if not setup_notebook(
    required_keys=[
        "GOOGLE_API_KEY",
    ]
):
    raise ValueError("Failed to setup notebook, please check your .env file")

masked_key_id = os.getenv("GOOGLE_API_KEY")[:5] + "*" * (
    len(os.getenv("GOOGLE_API_KEY")) - 5
)
print(f"Using Google API Key: {masked_key_id}")

In [2]:
from agent_platform.core.platforms.google.parameters import GooglePlatformParameters
from agent_platform.core.utils import SecretString

# Platform Parameters
parameters = GooglePlatformParameters(
    google_api_key=SecretString(os.getenv("GOOGLE_API_KEY")),
)

## Create a Platform Client

In [3]:
from agent_platform.core.platforms.google.client import GoogleClient

google_client = GoogleClient(
    parameters=parameters,
)

## Create a Tool

In [ ]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False


joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)


async def random_number_generator(
    min_value: Annotated[int, "The minimum value of the range"],
    max_value: Annotated[int, "The maximum value of the range"],
) -> int:
    """Generate a random number.

    Arguments:
        min_value: The minimum value of the range.
        max_value: The maximum value of the range.

    Returns:
        A random number between min_value and max_value.
    """
    import random

    return random.randint(min_value, max_value)


random_number_generator_tool = ToolDefinition.from_callable(random_number_generator)

print(random_number_generator_tool)

## Create a Prompt

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[
        PromptTextContent(
            text="Please use the joke_judge tool to "
            "judge if the following joke is funny: "
            '"Why did the chicken cross the road?"\n'
            '"To get to the other side!"\n'
            "Also, generate a random number between 1 and 100.\n"
            "Run these tools in parallel.",
        )
    ],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)

await general_prompt.finalize_messages()
google_prompt = await google_client.converters.convert_prompt(
    general_prompt,
    "gemini-2.5-flash-preview-04-17-low",
)

print(google_prompt)

## Request and Stream a Response

In [ ]:
deltas = []
i = 0
async for delta in google_client.generate_stream_response(
    google_prompt,
    "gemini-2.5-flash-preview-04-17-low",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [ ]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembled_dict = combine_generic_deltas(deltas)
pprint(assembled_dict)

### Now as a ResponseMessage

In [ ]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembled_dict)
pprint(response_message)

# Continue the Conversation

## Execute the Tools

In [ ]:
from agent_platform.core.responses import ResponseToolUseContent

print(response_message.content)

tool_use_content_block_joke_judge = response_message.content[0]
assert isinstance(tool_use_content_block_joke_judge, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block_joke_judge.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

tool_use_content_block_rng = response_message.content[1]
assert isinstance(tool_use_content_block_rng, ResponseToolUseContent)

random_number = await random_number_generator(
    int(tool_use_content_block_rng.tool_input["min_value"]),
    int(tool_use_content_block_rng.tool_input["max_value"]),
)

print(f"Generated Random Number: {random_number}")

## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content blocks of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block_joke_judge, ResponseToolUseContent)
assert isinstance(tool_use_content_block_rng, ResponseToolUseContent)

tool_result_content_joke_judge = PromptToolResultContent(
    tool_name=tool_use_content_block_joke_judge.tool_name,
    tool_call_id=tool_use_content_block_joke_judge.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

tool_result_content_rng = PromptToolResultContent(
    tool_name=tool_use_content_block_rng.tool_name,
    tool_call_id=tool_use_content_block_rng.tool_call_id,
    content=[PromptTextContent(text=str(random_number))],
)

print(tool_result_content_joke_judge)
print(tool_result_content_rng)

## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [ ]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use_joke_judge = response_message.content[0]
original_tool_use_rng = response_message.content[1]
assert isinstance(original_tool_use_joke_judge, ResponseToolUseContent)
assert isinstance(original_tool_use_rng, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use_joke_judge = ThreadToolUsageContent.from_response_tool_use(
    original_tool_use_joke_judge
)
prompt_tool_use_joke_judge = PromptToolUseContent.from_thread_tool_usage(
    ai_tool_use_joke_judge
)

ai_tool_use_rng = ThreadToolUsageContent.from_response_tool_use(original_tool_use_rng)
prompt_tool_use_rng = PromptToolUseContent.from_thread_tool_usage(ai_tool_use_rng)

pprint(prompt_tool_use_joke_judge)
pprint(prompt_tool_use_rng)

## Create new Prompt

In [ ]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content_joke_judge, tool_result_content_rng],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use_joke_judge, prompt_tool_use_rng],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool, random_number_generator_tool],
)
await return_gen_prompt.finalize_messages()
return_google_prompt = await google_client.converters.convert_prompt(
    return_gen_prompt,
    model_id="gemini-2.5-flash-preview-04-17-low",
)

pprint(return_google_prompt)

## Generate a new Response (Non-Streaming)

In [ ]:
response = await google_client.generate_response(
    return_google_prompt,
    "gemini-2.5-flash-preview-04-17-low",
)

pprint(response)